# 05 -- Sensitivity analysis (Technical Specification Step 5.3)

**SEA-FORWARD** OceanPrediction-A toolkit

Perturb the **atmospheric forcing** (wind amplitude in the GFS `for_croco`
files), **re-run CROCO**, and compare the **upwelling response**. This is
the clearest hands-on illustration in the whole toolkit of how the
OceanPrediction-A value chain is connected end to end:

```
   U3                      C1                       D1
Upstream forcing  --->  Core Forecasting  --->  Downstream diagnostic
(wind, perturbed          Engine (CROCO)          (upwelling index,
 here)                    re-run with the          SST response --
                          perturbed forcing)        computed here)
```

A change made at **U3** (the wind field edited below) only becomes visible
at **D1** (the SST/upwelling diagnostics at the end) *by passing through*
**C1** -- you cannot skip the model run. This is why Step 5.3 requires an
actual CROCO re-run between the two halves of this notebook, rather than
just perturbing a diagnostic directly.

**Prerequisite:** run `04_exercises.ipynb` first (or at least its Exercise
1) -- the Bakun upwelling index computation is reused unchanged below.


## Part A -- Perturb the wind forcing (U3)

We scale the 10 m wind components by a fixed amplitude factor (**x1.5**,
per Technical Specification Step 5.3). Wind *stress* in bulk-flux
formulations scales roughly with the square of wind speed, so a 1.5x
wind-*speed* perturbation is a substantially stronger forcing change than
it first appears -- worth keeping in mind when you look at the SST
response in Part C.

CROCO reads its atmospheric forcing straight out of
`downloaded_data/GFS/for_croco/` -- one file per variable (U-component,
V-component, radiation, humidity, ...). There is no single combined
"baseline forcing file" to edit and hand to a re-run under a different
name; instead:

1. **Back up the whole `for_croco/` directory** to a sibling
   `for_croco_origin/` -- every GFS variable, untouched, kept as the
   pristine record of what this cycle was actually forced with.
2. **Scale the U- and V-component wind files IN PLACE, inside
   `for_croco/`** -- the live directory CROCO's re-run (Part B) reads from
   -- so the re-run picks up the perturbation with no `--blk` override or
   separate run directory needed.

Every other GFS variable (radiation, humidity, pressure, precipitation,
...) is left completely unmodified -- only the two wind-component files are
touched, and only inside `for_croco/` (never inside `for_croco_origin/`,
which stays the pristine backup).


In [ ]:
import sys, os, glob, shutil
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

import sftools.postprocess as pp
import sftools.validation as val
import _paths


In [ ]:
CONFIG   = os.environ.get("SEAFORWARD_CONFIG", "Canary_12")
MAIN_DIR = os.path.expanduser(os.environ.get("SEAFORWARD_MAIN_DIR", "~/seaforward/forecast/model-runs"))
AVAILABLE_CYCLES = _paths.list_cycles(MAIN_DIR, CONFIG)
print(f"forecast cycles found under {os.path.join(MAIN_DIR, CONFIG)}: {AVAILABLE_CYCLES}")

# >>> SET THIS to the cycle you want to perturb, e.g. "20260711_plain" <<<
CYCLE = os.environ.get("SEAFORWARD_CYCLE", AVAILABLE_CYCLES[-1] if AVAILABLE_CYCLES else "")


In [ ]:
AMP_FACTOR = 1.5   # per Step 5.3 of the operational workflow

CROCO_HIS, REFERENCE, MAIN_DIR = _paths.get_paths(cycle=CYCLE, config=CONFIG, main_dir=MAIN_DIR)
YORIG = 2000   # forecast runs from the Copernicus Marine Forecast / Mercator anfc

CYCLE_DIR = _paths.cycle_dir(MAIN_DIR, CONFIG, CYCLE)
GFS_DIR        = os.path.join(CYCLE_DIR, "downloaded_data", "GFS", "for_croco")          # live forcing CROCO reads
GFS_BACKUP_DIR = os.path.join(CYCLE_DIR, "downloaded_data", "GFS", "for_croco_origin")   # pristine backup

print(f"Opening forecast cycle {CYCLE}")
print(f"  CROCO history (baseline, before rerun): {CROCO_HIS}")
print(f"  GFS forcing (live, will be perturbed) : {GFS_DIR}")
print(f"  GFS forcing backup (pristine)         : {GFS_BACKUP_DIR}")


In [ ]:
# CROCOTOOLS GFS forcing: one file per variable, named <VARIABLE>_Y..M...nc.
# Detect the U-/V-component wind files however many there are (usually one
# per forecast month, so typically just one of each for a 5-day cycle).
u_files = sorted(glob.glob(os.path.join(GFS_DIR, "U-COMPONENT_OF_WIND_*.nc")))
v_files = sorted(glob.glob(os.path.join(GFS_DIR, "V-COMPONENT_OF_WIND_*.nc")))
if not u_files or not v_files:
    raise FileNotFoundError(
        f"no U-/V-component wind files found under {GFS_DIR} -- "
        f"contents: {sorted(os.listdir(GFS_DIR)) if os.path.isdir(GFS_DIR) else '(directory not found)'}")

print(f"found {len(u_files)} U-component file(s): {[os.path.basename(f) for f in u_files]}")
print(f"found {len(v_files)} V-component file(s): {[os.path.basename(f) for f in v_files]}")


In [ ]:
# ---- 1) back up the WHOLE for_croco directory before touching anything ----
if os.path.exists(GFS_BACKUP_DIR):
    print(f"backup already exists at {GFS_BACKUP_DIR} -- not overwriting "
         "(delete it manually first if you want to redo the backup from the current for_croco/)")
else:
    shutil.copytree(GFS_DIR, GFS_BACKUP_DIR)
    print(f"backed up the original GFS forcing -> {GFS_BACKUP_DIR}")


In [ ]:
# ---- 2) scale the wind files IN PLACE, inside the live for_croco/ ----
def _wind_varname(ds):
    """The wind-speed-component variable in a CROCOTOOLS GFS file -- usually
    the only data variable, but fall back to matching on the name if not."""
    candidates = [v for v in ds.data_vars if "wind" in v.lower()]
    if len(candidates) == 1:
        return candidates[0]
    if len(ds.data_vars) == 1:
        return list(ds.data_vars)[0]
    raise ValueError(f"could not identify the wind variable among {list(ds.data_vars)}")

def _scale_wind_file(path, amp_factor):
    ds = xr.open_dataset(path)
    varname = _wind_varname(ds)
    scaled = ds[varname] * amp_factor
    scaled.attrs.update(ds[varname].attrs)
    ds[varname] = scaled
    ds.attrs["history"] = (ds.attrs.get("history", "") +
                           f" | SEA-FORWARD 05_sensitivity: wind x{amp_factor} ({varname}) for Step 5.3")
    tmp_path = path + ".tmp"
    ds.to_netcdf(tmp_path)
    ds.close()
    os.replace(tmp_path, path)   # only swap in the new file once the write has fully succeeded
    return varname

for path in u_files + v_files:
    varname = _scale_wind_file(path, AMP_FACTOR)
    print(f"  scaled {os.path.basename(path)} ({varname!r}) x{AMP_FACTOR}")

print(f"\n{len(u_files) + len(v_files)} file(s) perturbed in place under {GFS_DIR}")
print(f"(pristine originals kept at {GFS_BACKUP_DIR})")


## Part B -- Rename existing outputs, then re-run CROCO (C1)

The perturbed forcing from Part A is already sitting in `for_croco/` --
CROCO's re-run will pick it up with no further changes. But the re-run
writes its outputs (`croco_his.nc` and friends) into the *same*
`fcst/CROCO_FILES/` directory the baseline run already used -- so the
baseline files have to be moved out of the way FIRST, or the re-run would
silently overwrite them and there would be nothing left to compare against.

Every file in `fcst/CROCO_FILES/` is renamed `<name>_origin<ext>` (e.g.
`croco_his.nc` -> `croco_his_origin.nc`) before the re-run -- not just the
history file, since a full CROCO run can also write restart/average/
station files that would be just as silently lost otherwise.


In [ ]:
CROCO_FILES_DIR = os.path.dirname(CROCO_HIS)   # .../fcst/CROCO_FILES

renamed = []
for fname in sorted(os.listdir(CROCO_FILES_DIR)):
    src = os.path.join(CROCO_FILES_DIR, fname)
    if not os.path.isfile(src):
        continue
    stem, ext = os.path.splitext(fname)
    if stem.endswith("_origin"):
        continue   # already renamed -- safe to re-run this cell
    dst = os.path.join(CROCO_FILES_DIR, f"{stem}_origin{ext}")
    if os.path.exists(dst):
        print(f"  skip {fname}: {os.path.basename(dst)} already exists")
        continue
    os.rename(src, dst)
    renamed.append((fname, os.path.basename(dst)))

for old, new in renamed:
    print(f"  {old} -> {new}")
print(f"{len(renamed)} file(s) renamed in {CROCO_FILES_DIR}")


Now re-run CROCO for this cycle. This happens *outside* the notebook,
using the same forecast orchestration described in the Technical
Specification (Step 4 of the operational workflow) -- no `--blk` or
`--outdir` override needed, since the perturbed forcing is already in
place in `for_croco/` and the renamed baseline outputs above are out of
the way:

```bash
source ~/seaforward/env.sh
source ~/seaforward/forecast/track.sh
export CONFIG_NAME=Canary_12
export FCAST=${CROCO_RUNS_ROOT}/${CONFIG_NAME}     # forecast/scratch/Canary_12
cd ${FCAST}
conda deactivate                                    # run outside conda
./croco croco.in 2>&1 | tee run.log | tail -60
```

Once the run completes, `fcst/CROCO_FILES/croco_his.nc` is the perturbed
run's fresh output, and `croco_his_origin.nc` (renamed above) is the
baseline -- re-run the cell below to confirm both are present.


In [ ]:
HIS_BASELINE = os.path.join(CROCO_FILES_DIR, "croco_his_origin.nc")
HIS_PERTURBED = CROCO_HIS   # the re-run writes a fresh croco_his.nc here

assert os.path.exists(HIS_BASELINE), f"missing baseline history: {HIS_BASELINE} -- did the rename cell above run?"
assert os.path.exists(HIS_PERTURBED), (
    f"missing perturbed-run history: {HIS_PERTURBED}"
    " -> run Part B (the CROCO re-run) before continuing.")

print(f"  baseline : {HIS_BASELINE}")
print(f"  perturbed: {HIS_PERTURBED}")


## Part C -- Compare the upwelling response (D1)

Three comparisons, from the simplest to the most physically direct:

1. **SST difference map** (perturbed minus baseline): where did the wind
   change cool the surface, and by how much?
2. **Bakun upwelling index** at the coastal reference point (Exercise 1 of
   `04_exercises.ipynb`, reused verbatim), baseline vs. perturbed.
3. **Domain statistics** of the SST change, to put a single number on "how
   much stronger is upwelling with 1.5x wind".


In [ ]:
dsb_his = pp.open_history(HIS_BASELINE, Yorig=YORIG)
dsp_his = pp.open_history(HIS_PERTURBED, Yorig=YORIG)

clon, clat, cmask = pp.lonlatmask(dsb_his)
sst_baseline = pp.surface(dsb_his, "temp", tindex=-1).values
sst_perturbed = pp.surface(dsp_his, "temp", tindex=-1).values
sst_diff = sst_perturbed - sst_baseline

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
vmin, vmax = np.nanmin([sst_baseline, sst_perturbed]), np.nanmax([sst_baseline, sst_perturbed])
for ax, f, title in zip(axes, (sst_baseline, sst_perturbed, sst_diff),
                        ("baseline SST", f"perturbed SST (wind x{AMP_FACTOR})",
                         "difference (perturbed - baseline)")):
    if f is sst_diff:
        dmax = np.nanpercentile(np.abs(f[np.isfinite(f)]), 98)
        h = ax.pcolormesh(clon, clat, f, cmap="RdBu_r", vmin=-dmax, vmax=dmax, shading="auto")
    else:
        h = ax.pcolormesh(clon, clat, f, cmap="RdYlBu_r", vmin=vmin, vmax=vmax, shading="auto")
    ax.set_title(title, fontsize=10)
    plt.colorbar(h, ax=ax, shrink=0.8, label="degC")
fig.suptitle(f"CROCO SST response to a {AMP_FACTOR}x wind-amplitude perturbation")
plt.tight_layout(); plt.show()


In [ ]:
sst_change = val.domain_statistics(sst_perturbed, sst_baseline)
print(f"SST change (perturbed vs. baseline): mean = {sst_change['bias']:+.3f} C, "
     f"RMS = {sst_change['rmse']:.3f} C, n = {sst_change['n']}")
print("A negative mean bias here is the expected upwelling signature: stronger "
     "upwelling-favourable wind -> more coastal cooling.")


### Bakun upwelling index -- baseline vs. perturbed

Reusing the Exercise 1 calculation from `04_exercises.ipynb` unchanged,
applied to both wind fields, so the *only* thing that differs between the
two numbers below is the `AMP_FACTOR` scaling applied in Part A. The
baseline wind is read from `for_croco_origin/` (the pristine backup) --
never from the live `for_croco/`, which has been perturbed in place since
Part A.


In [ ]:
OMEGA = 7.2921e-5
RHO_AIR, RHO_WATER, CD = 1.22, 1025.0, 1.3e-3
COAST_ANGLE_DEG = 0.0   # TODO: same coastline angle used in 04_exercises.ipynb


def bakun_index(u10, v10, lat0, coast_angle_deg=COAST_ANGLE_DEG):
    theta = np.deg2rad(coast_angle_deg)
    w_along = u10 * np.cos(theta) + v10 * np.sin(theta)
    w_speed = np.sqrt(u10 ** 2 + v10 ** 2)
    f = 2 * OMEGA * np.sin(np.deg2rad(lat0))
    return (RHO_AIR * CD * w_speed * w_along) / (RHO_WATER * f)


LON0 = float(clon[cmask > 0][0]); LAT0 = float(clat[cmask > 0][0])

# baseline wind, read from the pristine backup (for_croco_origin) -- the
# live for_croco/ files were scaled in place in Part A, so reading the
# "baseline" from there now would already be the perturbed value.
u_backup = sorted(glob.glob(os.path.join(GFS_BACKUP_DIR, "U-COMPONENT_OF_WIND_*.nc")))
v_backup = sorted(glob.glob(os.path.join(GFS_BACKUP_DIR, "V-COMPONENT_OF_WIND_*.nc")))
dsb_blk = xr.open_dataset(u_backup[0])
dsv_blk = xr.open_dataset(v_backup[0])
uname, vname = _wind_varname(dsb_blk), _wind_varname(dsv_blk)
j = np.argmin((dsb_blk["lat"].values[:, 0] - LAT0) ** 2)
i = np.argmin((dsb_blk["lon"].values[0, :] - LON0) ** 2)
u0_base = float(dsb_blk[uname].isel(time=-1).values[j, i])
v0_base = float(dsv_blk[vname].isel(time=-1).values[j, i])
u0_pert = u0_base * AMP_FACTOR
v0_pert = v0_base * AMP_FACTOR
dsb_blk.close(); dsv_blk.close()

Qx_base = bakun_index(u0_base, v0_base, LAT0)
Qx_pert = bakun_index(u0_pert, v0_pert, LAT0)
print(f"Bakun index, baseline : {Qx_base:+.3f} m2/s")
print(f"Bakun index, perturbed: {Qx_pert:+.3f} m2/s")
if Qx_base != 0:
    print(f"  ratio: x{Qx_pert / Qx_base:.2f} relative to baseline")


> **Note.** The qualitative result -- that the index scales *faster* than
> linearly with `AMP_FACTOR` -- holds because both the alongshore-wind term
> *and* the wind-speed term in the Bakun formula grow together (Qx is
> proportional to `|W| * W_alongshore`, i.e. roughly quadratic in wind
> speed for wind blowing mostly alongshore). Compare `Qx_pert/Qx_base`
> above to `AMP_FACTOR**2` to check this directly on your own run.


In [ ]:
# Self-check: the perturbed index should scale up with AMP_FACTOR, in the
# same direction as the baseline (same upwelling/downwelling sign)
assert np.sign(Qx_pert) == np.sign(Qx_base), "perturbation flipped the upwelling sign -- check COAST_ANGLE_DEG"
assert abs(Qx_pert) > abs(Qx_base), "perturbed index should be stronger than baseline for AMP_FACTOR > 1"
print("self-check passed")

dsb_his.close(); dsp_his.close()


---
### Summary

This notebook closed the loop from **U3** (perturbed wind forcing) through
**C1** (the re-run CROCO model) to **D1** (the SST and upwelling-index
response) -- the exact chain the Technical Specification's Data
Consistency Chain (DCC) architecture requires. Record your `CYCLE`,
`AMP_FACTOR`, the resulting SST bias/RMSE, and the Bakun-index ratio in
your lab notes.
